# Loop-tendril publication figures

Run from `examples/tendril_regression/` with the `examples` extra and LaTeX installed. First generate the inputs with `python generate_loop_tendril_point_clouds.py`, then `python featurize_point_clouds.py` (these steps and the regression are computationally expensive).

This notebook writes the regression and graph showcase figures to `outputs/`. Regression inputs are `outputs/features_single_uniform/<label>.csv`: one row per sample, with `sample_id`, `root_mode`, `tendril_mode`, the two targets, and numeric `f0`, `f1`, … feature columns. Input generation settings and seeds are recorded in the adjacent metadata JSON files.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RepeatedKFold

from loop_tendrils import generate_parametric_loop_tendrils

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 300,
    "axes.grid": False,
    "grid.color": "#d3d3d3",
    "grid.linewidth": 0.5,
    "axes.linewidth": 0.8,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

width = 448.13095 / 72.27  # Manuscript line width in inches.


## Graph showcase

Three seeded examples span the input parameter ranges. Labels show generator lengths before persistence-death normalization (seed `4`; single tendrils with uniformly distributed roots).


In [ ]:
showcase_specs = [
    {"n_tendrils": 4, "tendril_length": 0.25},
    {"n_tendrils": 10, "tendril_length": 0.38},
    {"n_tendrils": 18, "tendril_length": 0.12},
]
showcase_seed = 4

fig, axes = plt.subplots(
    1,
    len(showcase_specs),
    figsize=(width, width*0.3),
    constrained_layout=True,
)

for ax, spec in zip(axes, showcase_specs):
    graph = generate_parametric_loop_tendrils(
        **spec,
        tendril_mode="single",
        root_mode="uniform",
        branching_probability=0.0,
        n_loop_vertices=80,
        loop_noise=0.01,
        normalize=False,
        seed=showcase_seed,
    )

    nodes = np.asarray(graph.nodes)
    edges = np.asarray(graph.edges)
    tendril_edge_indices = np.asarray(
        graph.labels["tendril_edge_indices"],
        dtype=int,
    )

    ax.add_collection(
        LineCollection(
            nodes[edges],
            colors="#7a7f86",
            linewidths=1.1,
            capstyle="round",
            joinstyle="round",
            zorder=1,
        )
    )
    ax.add_collection(
        LineCollection(
            nodes[edges[tendril_edge_indices]],
            colors="#d95f02",
            linewidths=2.0,
            capstyle="round",
            joinstyle="round",
            zorder=2,
        )
    )
    ax.scatter(
        nodes[:, 0],
        nodes[:, 1],
        s=3,
        color="#1f2933",
        linewidths=0,
        zorder=3,
    )
    ax.autoscale()
    ax.set_aspect("equal", adjustable="box")
    ax.margins(0.08)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(
        f"{spec['n_tendrils']} tendrils of length {spec['tendril_length']:.2f}",
        fontsize=11,
    )

fig.savefig(output_dir / "loop_tendril_graph_showcase.pdf", bbox_inches="tight")
fig.savefig(
    output_dir / "loop_tendril_graph_showcase.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Regression

Five-fold cross-validation is repeated ten times with identical splits across representations (split seed `0`; forest seed `0`, 500 trees, minimum leaf size `2`). Scatter points show out-of-fold predictions from the **first repeat**; annotations report mean ± sample standard deviation across **all 50 test folds**. Fold SD is descriptive variability, not a confidence interval.


In [ ]:
feature_dir = output_dir / "features_single_uniform"
labels = [
    "standard_landscapes",
    "unsigned_excess_curvature",
    "signed_excess_curvature",
    "tendril_count",
    "avg_tendril_length",
]
targets = ["n_tendrils", "tendril_length"]
mode_filter = {"tendril_mode": ["single"], "root_mode": ["uniform"]}
active_mode_filter = "single_uniform"
n_splits = 5
n_repeats = 10
cv_random_state = 0
rf_random_state = 0

missing_feature_paths = [feature_dir / f"{label}.csv" for label in labels if not (feature_dir / f"{label}.csv").exists()]
if missing_feature_paths:
    missing_paths = "\n".join(str(path) for path in missing_feature_paths)
    raise FileNotFoundError(
        "Missing feature CSVs. Generate them with featurize_point_clouds.py.\n"
        f"Missing paths:\n{missing_paths}"
    )

feature_frames = {}
feature_columns_by_label = {}
for label in labels:
    df = pd.read_csv(feature_dir / f"{label}.csv")
    for mode_column, allowed_values in mode_filter.items():
        if mode_column not in df.columns:
            raise ValueError(f"Column {mode_column!r} not found in {feature_dir / f'{label}.csv'}")
        df = df[df[mode_column].isin(allowed_values)].copy()
    if len(df) < n_splits:
        raise ValueError(
            f"Mode filter {active_mode_filter!r} leaves only {len(df)} rows in {feature_dir / f'{label}.csv'}"
        )
    if df["sample_id"].duplicated().any():
        raise ValueError(f"Duplicate sample IDs in {feature_dir / f'{label}.csv'}")
    df = df.sort_values("sample_id").reset_index(drop=True)

    feature_columns = [column for column in df.columns if column.startswith("f") and column[1:].isdigit()]
    if not feature_columns:
        raise ValueError(f"No feature columns found in {feature_dir / f'{label}.csv'}")

    feature_frames[label] = df
    feature_columns_by_label[label] = feature_columns

reference_df = feature_frames[labels[0]]
reference_ids = reference_df["sample_id"].to_numpy()
for label, df in feature_frames.items():
    if not np.array_equal(df["sample_id"].to_numpy(), reference_ids):
        raise ValueError(f"Sample IDs for {label!r} do not match the reference feature file")
    for target in targets:
        if not np.allclose(df[target].to_numpy(), reference_df[target].to_numpy()):
            raise ValueError(f"Target {target!r} for {label!r} does not match the reference feature file")

cv = RepeatedKFold(
    n_splits=n_splits,
    n_repeats=n_repeats,
    random_state=cv_random_state,
)
cv_splits = list(cv.split(reference_ids))

rf_fold_metrics = []
rf_predictions = {}

for label in labels:
    df = feature_frames[label]
    feature_columns = feature_columns_by_label[label]
    X = df[feature_columns].to_numpy()

    for target in targets:
        y = df[target].to_numpy()
        first_repeat_predictions = np.full(len(y), np.nan)

        for split_index, (train_indices, test_indices) in enumerate(cv_splits):

            rf_regressor = RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=2,
                random_state=rf_random_state,
                n_jobs=-1,
            )
            rf_regressor.fit(X[train_indices], y[train_indices])
            y_pred = rf_regressor.predict(X[test_indices])
            y_test = y[test_indices]

            if split_index < n_splits:
                first_repeat_predictions[test_indices] = y_pred

            rf_fold_metrics.append(
                {
                    "feature_label": label,
                    "target": target,
                    "mode_filter": active_mode_filter,
                    "split_index": split_index,
                    "repeat": split_index // n_splits,
                    "fold": split_index % n_splits,
                    "n_samples": len(df),
                    "n_features": len(feature_columns),
                    "n_train": len(train_indices),
                    "n_test": len(test_indices),
                    "mae": mean_absolute_error(y_test, y_pred),
                    "rmse": mean_squared_error(y_test, y_pred) ** 0.5,
                    "r2": r2_score(y_test, y_pred),
                }
            )

        if np.isnan(first_repeat_predictions).any():
            raise RuntimeError("The first CV repeat did not predict every sample exactly once")
        rf_predictions[(label, target)] = (y, first_repeat_predictions)

rf_fold_metrics = pd.DataFrame(rf_fold_metrics)
rf_regression_metrics = (
    rf_fold_metrics.groupby(["feature_label", "target", "mode_filter"], as_index=False)
    .agg(
        n_samples=("n_samples", "first"),
        n_features=("n_features", "first"),
        n_test_folds=("split_index", "count"),
        mae_mean=("mae", "mean"),
        mae_sd=("mae", "std"),
        rmse_mean=("rmse", "mean"),
        rmse_sd=("rmse", "std"),
        r2_mean=("r2", "mean"),
        r2_sd=("r2", "std"),
    )
    .sort_values(["target", "feature_label"])
    .reset_index(drop=True)
)


In [ ]:
publication_labels = labels
publication_targets = targets
column_xy_max = {
    "n_tendrils": 27.0,
    "tendril_length": 0.45,
}

feature_titles = {
    "standard_landscapes": "Standard persistence\nlandscapes",
    "unsigned_excess_curvature": "Measurement landscapes\nw.r.t. excess curvature\n(unsigned)",
    "signed_excess_curvature": "Measurement landscapes\nw.r.t. excess curvature\n(signed)",
    "tendril_count": "Measurement landscapes\nw.r.t. purely signed \nconnected components\n(signed)",
    "avg_tendril_length": "Measurement landscapes\nw.r.t. average length \nof purely signed\nconnected components\n(signed)",
}
target_titles = {
    "n_tendrils": "Number of tendrils",
    "tendril_length": "Tendril length",
}

missing_predictions = [
    (feature_label, target)
    for feature_label in publication_labels
    for target in publication_targets
    if (feature_label, target) not in rf_predictions
]
if missing_predictions:
    raise KeyError(f"Missing random forest predictions for: {missing_predictions}")

metric_lookup = rf_regression_metrics.set_index(["feature_label", "target"])



n_rows = len(publication_labels)
n_cols = len(publication_targets)
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(width, 0.75 * width * (1.7 * n_rows + 0.9) / 7.2),
    sharex="col",
    sharey="col",
    squeeze=False,
)
fig.subplots_adjust(left=0.4, right=0.965, bottom=0.05, top=0.915, wspace=0.16, hspace=0.15)

column_limits = {}
for target in publication_targets:
    values = []
    for feature_label in publication_labels:
        y_true, y_pred = rf_predictions[(feature_label, target)]
        values.extend([np.asarray(y_true), np.asarray(y_pred)])
    values = np.concatenate(values)
    low = float(np.nanmin(values))
    data_high = float(np.nanmax(values))
    padding = 0.06 * (data_high - low) if data_high > low else 1.0
    configured_high = float(column_xy_max[target])
    if configured_high < data_high:
        raise ValueError(
            f"Configured axis maximum {configured_high} for {target!r} would clip data up to {data_high}"
        )
    column_limits[target] = (low - padding, configured_high)

point_color = "#2f6f9f"
identity_color = "#202020"
grid_color = "#e8edf2"

for row_index, feature_label in enumerate(publication_labels):
    for col_index, target in enumerate(publication_targets):
        ax = axes[row_index, col_index]
        y_true, y_pred = rf_predictions[(feature_label, target)]
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        low, high = column_limits[target]

        ax.scatter(
            y_true,
            y_pred,
            s=10,
            color=point_color,
            edgecolor="white",
            linewidth=0.35,
            alpha=0.78,
            zorder=3,
        )
        ax.plot([low, high], [low, high], color=identity_color, linestyle=(0, (4, 3)), linewidth=1.0, zorder=2)
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_box_aspect(1)
        ax.grid(True, color=grid_color, linewidth=0.75)
        ax.set_axisbelow(True)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4, prune=None))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune=None))

        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)
        for spine in ["left", "bottom"]:
            ax.spines[spine].set_color("#8a949e")
            ax.spines[spine].set_linewidth(0.8)

        metric_row = metric_lookup.loc[(feature_label, target)]
        rmse_decimals = 3 if target == "n_tendrils" else 4
        ax.text(
            0.04,
            0.94,
            (
                f"R2 = {metric_row['r2_mean']:.3f} $\\pm$ {metric_row['r2_sd']:.3f}\n"
                f"RMSE = {metric_row['rmse_mean']:.{rmse_decimals}f} $\\pm$ "
                f"{metric_row['rmse_sd']:.{rmse_decimals}f}"
            ),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7,
            color="#30343b",
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "#d8dee6", "alpha": 0.88},
        )

        if row_index == n_rows - 1:
            ax.set_xlabel("True value")
        else:
            ax.tick_params(labelbottom=False)
        if col_index == 0:
            ax.set_ylabel("Predicted value")

fig.canvas.draw()

for col_index, target in enumerate(publication_targets):
    position = axes[0, col_index].get_position()
    fig.text(
        (position.x0 + position.x1) / 2,
        0.92,
        target_titles.get(target, target.replace("_", " ").title()),
        ha="center",
        va="bottom",
        color="#1f2933",
        fontsize = 10,
    )

for row_index, feature_label in enumerate(publication_labels):
    position = axes[row_index, 0].get_position()
    fig.text(
        0.07,
        (position.y0 + position.y1) / 2,
        feature_titles.get(feature_label, feature_label.replace("_", " ").title()),
        ha="left",
        va="center",
        color="#1f2933",
    )

fig.suptitle(
    "Random forest regression",
    x=0.07,
    y=0.986,
    ha="left",
    fontweight="bold",
)
fig.text(
    0.07,
    0.955,
    "10 $\\times$ 5-fold cross-validation; metrics: mean $\\pm$ split SD",
    ha="left",
    va="top",
    fontsize=8,
    color="#4b5563",
)
fig.legend(
    handles=[
        Line2D([0], [0], marker="o", color="none", markerfacecolor=point_color, markeredgecolor="white", markersize=6, label="random forest prediction"),
        Line2D([0], [0], color=identity_color, linestyle=(0, (4, 3)), linewidth=1.0, label="perfect prediction"),
    ],
    loc="upper right",
    bbox_to_anchor=(0.955, 1),
    frameon=False,
)

fig.savefig(output_dir / "tendril_regression.pdf", dpi=300)
fig.savefig(output_dir / "single_uniform_regression_plot.png", dpi=300)
plt.show()